# Optuna Trial Analysis

Use this notebook after running `scripts/run_optuna_experiments.py` to inspect the stored trials.

In [1]:
from pathlib import Path

# Update these paths if you changed the defaults in `scripts/run_optuna_experiments.py`
# CSV_PATH = Path('../results/optuna_a2_trials.csv')
# STORAGE_PATH = Path('../results/optuna_a2.db')

CSV_PATH = Path('../results/efficient_attn_sweep.csv')
STORAGE_PATH = Path('../results/efficient_attn_sweep.db')


print(f'CSV path: {CSV_PATH.resolve()}')
print(f'Study storage: {STORAGE_PATH.resolve()}')

CSV path: /home/tressen-arms/Documents/experiments/Action-Prior-Alignment/results/efficient_attn_sweep.csv
Study storage: /home/tressen-arms/Documents/experiments/Action-Prior-Alignment/results/efficient_attn_sweep.db


In [2]:
import pandas as pd

trials_df = pd.read_csv(CSV_PATH)
display(trials_df.head())
print(f'Total trials loaded: {len(trials_df)}')


,number,value,state,params_adjust_lr,params_efficient_attn,params_heads,params_hidden_size,params_lang_emb,params_layers,params_lr,params_normalize,params_step_ratio,params_step_size,params_use_rope,params_width,user_attrs_evaluated_samples,user_attrs_failure,user_attrs_test_accuracy,user_attrs_train_history,duration
0,0,NaN,FAIL,True,efficient,4,384,True,2,0.000028,False,0.685363,50.0,True,768,NaN,NaN,NaN,NaN,0 days 00:00:48.292852
1,1,7.534681,COMPLETE,False,hydra,4,512,True,3,0.000013,True,NaN,NaN,True,768,1332.0,NaN,0.254505,"[2.947475122219012, 2.628800455053339, 2.45689...",0 days 00:17:51.639688
2,2,2.619416,COMPLETE,True,pola,8,384,True,1,0.000036,False,0.156261,50.0,False,640,1332.0,NaN,0.240240,"[3.1244278161918095, 2.990403085549971, 2.9318...",0 days 00:10:57.496844
3,3,2.622029,COMPLETE,True,mobile,4,384,False,1,0.000066,False,0.477642,50.0,False,768,1332.0,NaN,0.237237,"[3.1521366507704163, 2.9645875517049443, 2.841...",0 days 00:05:04.020422
4,4,NaN,PRUNED,False,pola,8,256,True,3,0.000160,False,NaN,NaN,True,640,NaN,The size of tensor a (640) must match the size...,NaN,NaN,0 days 00:00:02.739660


Total trials loaded: 101


In [11]:
import importlib.util
module_missing = importlib.util.find_spec('optuna') is None
assert not module_missing, 'Optuna with visualization extras is required.'

import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
)
import plotly.io as pio

pio.renderers.default = 'notebook_connected'

study = optuna.load_study(study_name='efficient_attn_sweep', storage=f'sqlite:///{STORAGE_PATH}')
print(f'Best value: {study.best_trial.value:.4f}')
print('Best params:')
for key, value in study.best_trial.params.items():
    print(f'  {key}: {value}')


Best value: 2.0297
Best params:
  lr: 3.7974187279446955e-05
  hidden_size: 256
  layers: 3
  width: 768
  heads: 8
  efficient_attn: mobile
  lang_emb: True
  normalize: False
  use_rope: False
  adjust_lr: True
  step_size: 45
  step_ratio: 0.48321948247053237


In [5]:
study.best_params

{'lr': 3.7974187279446955e-05,
 'hidden_size': 256,
 'layers': 3,
 'width': 768,
 'heads': 8,
 'efficient_attn': 'mobile',
 'lang_emb': True,
 'normalize': False,
 'use_rope': False,
 'adjust_lr': True,
 'step_size': 45,
 'step_ratio': 0.48321948247053237}

In [12]:
from IPython.display import display

fig = plot_optimization_history(study)
display(fig)

fig = plot_param_importances(study)
display(fig)

fig = plot_parallel_coordinate(study)
display(fig)

fig = plot_slice(study)
display(fig)


## Diverse top trials

In [10]:
study

In [14]:
import numpy as np
import pandas as pd

# ------------------------ configurable knobs ------------------------
N_BEST = 5
MIN_DISTANCE = 0.25
TOP_K = 100  # consider top-K by value first (quality filter)

# Optional feature weights (default = 1.0). Add or tweak as you like.
FEATURE_WEIGHTS = {
    # 'params_efficient_attn': 1.5,
    # 'params_hidden_size': 0.7,
    # 'params_lr': 1.2,
}

# Force some columns to categorical regardless of dtype (e.g., small discrete numerics or enums)
FORCE_CATEGORICAL = {
    # 'params_efficient_attn',
    # 'params_hidden_size',
    # 'params_heads',
}

# Log-scale certain numeric parameters before distance (helps LR, weight decay, etc.)
LOG_SCALE_COLS = {
    'params_lr',
}

# If an integer-like numeric column has <= this many unique values, treat it as categorical unless overridden.
DISCRETE_AS_CATEGORICAL_MAX_UNIQUE = 10
# --------------------------------------------------------------------

def _is_integer_like(series: pd.Series) -> bool:
    return pd.api.types.is_integer_dtype(series) or (
        pd.api.types.is_numeric_dtype(series) and np.all(np.mod(series.dropna(), 1) == 0)
    )

def _infer_feature_types(df: pd.DataFrame) -> dict:
    """
    Return a dict: col -> {'type': 'numeric'|'categorical', 'weight': float}
    - bools treated as categorical
    - ints with few unique values -> categorical (unless overridden)
    """
    info = {}
    for col in df.columns:
        s = df[col]
        w = float(FEATURE_WEIGHTS.get(col, 1.0))

        if col in FORCE_CATEGORICAL:
            info[col] = {'type': 'categorical', 'weight': w}
            continue

        if pd.api.types.is_bool_dtype(s):
            info[col] = {'type': 'categorical', 'weight': w}
            continue

        if pd.api.types.is_numeric_dtype(s):
            # small discrete integer domains behave better as categories
            if _is_integer_like(s) and s.nunique(dropna=True) <= DISCRETE_AS_CATEGORICAL_MAX_UNIQUE:
                info[col] = {'type': 'categorical', 'weight': w}
            else:
                info[col] = {'type': 'numeric', 'weight': w}
            continue

        # strings/objects -> categorical
        info[col] = {'type': 'categorical', 'weight': w}

    return info

def _apply_pretransforms(df: pd.DataFrame, feature_info: dict) -> pd.DataFrame:
    df = df.copy()
    # log-scale selected numeric columns (safe only for positive values)
    for col in LOG_SCALE_COLS:
        if col in df.columns and feature_info.get(col, {}).get('type') == 'numeric':
            vals = df[col].astype(float)
            # handle non-positive or missing robustly
            with np.errstate(divide='ignore', invalid='ignore'):
                df[col] = np.where(vals > 0, np.log10(vals), np.nan)
    return df

def gower_distance_matrix(df: pd.DataFrame, feature_info: dict) -> np.ndarray:
    """
    Compute Gower-like distance for mixed data with weights and missing support.
    d(i,j) = sum_k w_k * d_k(i,j) / sum_k w_k over non-missing pairs
    where:
      - numeric: |x_i - x_j| / range_k (range 1 if degenerate)
      - categorical/bool: 0 if equal, 1 if not
    Missing for a feature excludes it from that pair's denominator.
    """
    n = len(df)
    if n == 0:
        return np.zeros((0, 0), dtype=float)

    # Prepare per-column arrays and masks
    arrays = {}
    masks = {}
    types = {}
    weights = {}

    # Precompute numeric ranges
    ranges = {}

    for col, meta in feature_info.items():
        s = df[col]
        types[col] = meta['type']
        weights[col] = float(meta['weight'])

        if meta['type'] == 'numeric':
            vals = pd.to_numeric(s, errors='coerce').astype(float)
            arrays[col] = vals.to_numpy()
            m = ~np.isnan(arrays[col])
            masks[col] = m
            vmin = np.nanmin(arrays[col]) if np.any(m) else 0.0
            vmax = np.nanmax(arrays[col]) if np.any(m) else 0.0
            rng = (vmax - vmin) if (vmax - vmin) > 0 else 1.0
            ranges[col] = float(rng)
        else:
            # categorical: keep as object; NaN marks missing
            arrays[col] = s.astype(object).to_numpy()
            masks[col] = ~pd.isna(arrays[col])

    D_num = np.zeros((n, n), dtype=float)   # numerator accumulator
    D_den = np.zeros((n, n), dtype=float)   # denominator accumulator

    idx = np.arange(n)
    for col in df.columns:
        t = types[col]
        w = weights[col]
        if w == 0.0:
            continue  # skip zero-weight features entirely
        a = arrays[col]
        m = masks[col]

        # pairwise mask: both non-missing
        mm = np.logical_and.outer(m, m)

        if t == 'numeric':
            # |x_i - x_j| / range
            diff = np.abs(a[:, None] - a[None, :]) / ranges[col]
            contrib = w * diff
        else:
            # categorical/boolean: 0 if equal, 1 if different
            # Equality comparison safely handles objects/strings/bools
            eq = (a[:, None] == a[None, :])
            diff = (~eq).astype(float)
            contrib = w * diff

        # accumulate only where both present
        D_num += np.where(mm, contrib, 0.0)
        D_den += np.where(mm, w, 0.0)

    # Normalize; where denominator == 0 (everything missing), set distance 0
    with np.errstate(divide='ignore', invalid='ignore'):
        D = np.where(D_den > 0, D_num / D_den, 0.0)

    # Make perfectly symmetric and zero diagonal
    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)
    return D

def greedy_diverse_selection(distance_matrix: np.ndarray,
                             ordered_indices: list,
                             n_best: int,
                             min_distance: float) -> list:
    """
    Greedy selection that respects min_distance if possible.
    Falls back to farthest-point sampling to fill the remainder.
    """
    selected = []
    for i in ordered_indices:
        if not selected:
            selected.append(i)
            if len(selected) == n_best:
                return selected
            continue
        dists = distance_matrix[i, selected]
        if np.all(dists >= min_distance):
            selected.append(i)
            if len(selected) == n_best:
                return selected

    # If we didn't reach n_best, fill via farthest-point (maximizing min distance)
    remaining = [i for i in ordered_indices if i not in selected]
    while len(selected) < n_best and remaining:
        # pick candidate that maximizes its minimum distance to current selection
        best_i = None
        best_score = -1.0
        for j in remaining:
            if selected:
                score = float(distance_matrix[j, selected].min())
            else:
                score = np.inf
            if score > best_score:
                best_score = score
                best_i = j
        selected.append(best_i)
        remaining.remove(best_i)

    return selected[:n_best]

# --------------------- main: build dataframe & select ---------------------
raw_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))

completed = raw_df[raw_df['state'] == 'COMPLETE'].copy().reset_index(drop=True)
assert not completed.empty, 'No completed trials available.'

# Sort by value respecting study direction if available
ascending = True
try:
    # optuna>=3: study.directions for multi-objective; fall back to .direction
    direction = getattr(study, 'direction', None)
    if direction is None and hasattr(study, 'directions'):
        direction = study.directions[0]
    if str(direction).upper().endswith('MAXIMIZE'):
        ascending = False
except Exception:
    pass

completed = completed.sort_values('value', ascending=ascending).reset_index(drop=True)

param_cols = sorted([c for c in completed.columns if c.startswith('params_') or c.startswith('param_')])
assert param_cols, 'No hyperparameter columns found in the dataframe.'

param_df = completed[param_cols].copy()

# Infer feature types and apply pre-transforms (e.g., log for LR)
feature_info = _infer_feature_types(param_df)
param_df = _apply_pretransforms(param_df, feature_info)

# (Optional) shrink to top-K by value for quality, then do diversity within this pool
pool = completed.head(min(TOP_K, len(completed))).copy()
param_pool = param_df.loc[pool.index]

# Build pairwise Gower-like distances with weights
D = gower_distance_matrix(param_pool[param_cols], feature_info)

# Order candidates by objective value (already sorted) and select diverse set
ordered_indices = list(range(len(pool)))  # 0..pool_size-1 in sorted order
selected_rel = greedy_diverse_selection(D, ordered_indices, N_BEST, MIN_DISTANCE)

# Map back to original completed indices
selected_abs = pool.iloc[selected_rel].index.tolist()
diverse_trials = completed.loc[selected_abs].copy()

# Report diagnostic: min pairwise distance among the selected set
if len(selected_rel) > 1:
    sel_d = D[np.ix_(selected_rel, selected_rel)]
    np.fill_diagonal(sel_d, np.inf)
    min_pairwise = float(np.min(sel_d))
else:
    min_pairwise = 0.0

print(f"Selected {len(diverse_trials)} trials. Target min distance={MIN_DISTANCE:.3f}. "
      f"Achieved min pairwise distance={min_pairwise:.3f}.")
display(diverse_trials[['number', 'value'] + param_cols])

Selected 5 trials. Target min distance=0.250. Achieved min pairwise distance=0.331.


,number,value,params_adjust_lr,params_efficient_attn,params_heads,params_hidden_size,params_lang_emb,params_layers,params_lr,params_normalize,params_step_ratio,params_step_size,params_use_rope,params_width
0,98,2.029700,True,mobile,8,256,True,3,0.000038,False,0.483219,45.0,False,768
4,33,2.054781,False,mobile,4,512,True,3,0.000071,True,NaN,NaN,False,768
18,13,2.101935,True,none,4,384,True,1,0.000137,False,0.144732,25.0,False,640
21,16,2.128569,False,none,4,256,True,2,0.000098,True,NaN,NaN,False,512
26,29,2.258005,True,efficient,4,256,True,1,0.000127,False,0.800682,35.0,False,640
